# 좌우 상성(platoon split) 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 `is_same_hand`(투수/타자 손 일치 여부)와
`hand_matchup`(둘의 조합 범주형, 4종)이 LightGBM 성능을 개선하는지 확인합니다.

**사전 검증 결과** (구현 전에 확인함) - `pitcher_hand`×`batter_hand` 조합별 실제 성공률:

| pitcher_hand | batter_hand | 성공률 | 표본수 |
|---|---|---|---|
| 1 | 1 | 0.4909 | 170,292 |
| 1 | 2 | 0.5375 | 211,059 |
| 2 | 1 | 0.5307 | 525,455 |
| 2 | 2 | 0.5221 | 568,286 |

최대 4.7%p 차이가 나고 표본 수도 충분해서(최소 17만행) 우연은 아닐 가능성이 높습니다.
다만 `pitcher_hand`/`batter_hand`가 이미 개별 범주형 피처로 들어가 있어서, 트리가
두 컬럼을 순차 분기로 조합해 이미 이 정보를 암묵적으로 잡아냈을 수도 있습니다
(맞대결 피처 때 겪은 것과 같은 함정 - 카디널리티가 낮아서(2×2=4가지뿐) 트리가
알아서 찾기 쉬운 경우라 명시적 피처의 실질적 효과는 작을 수 있음).

**비교 기준선**:
- LightGBM 20만행 OOF Brier: **0.245666**
- LightGBM 전체(147만행) OOF Brier: **0.244139**

⚠️ 맞대결 피처 실험에서 20만행 결과(+0.043%)와 전체 데이터 결과(-0.068%)의 부호가
뒤집혔던 걸 기억하고, 이번에도 20만행 결과는 참고용으로만 보고 **전체 데이터 결과로
최종 판단**합니다.

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (좌우 상성 피처 포함)

`use_matchup_feature`는 기본값 False라 맞대결 피처는 자동으로 빠집니다.
`is_same_hand`, `hand_matchup`은 기본으로 항상 포함됩니다.

In [2]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(train, None)
cat_features = [c for c in CAT_COLS if c in feat_cols]

platoon_cols = ["is_same_hand", "hand_matchup"]
print(f"피처 개수: {len(feat_cols)} (좌우 상성 피처: {platoon_cols})")
print(f"cat_features: {cat_features}")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

(1475092, 49)
피처 개수: 71 (좌우 상성 피처: ['is_same_hand', 'hand_matchup'])
cat_features: ['top_bottom', 'game_type', 'base_state', 'pitcher_hand', 'batter_hand', 'hand_matchup']
기준(무정보) Brier = 0.24944


## 2. 20만행 샘플로 빠른 확인 (참고용 - 결정적 근거로 쓰지 않음)

In [3]:
SAMPLE_N = 200_000

sample_idx = train_feat.sample(n=min(SAMPLE_N, len(train_feat)), random_state=42).index
X_small = train_feat.loc[sample_idx, feat_cols]
y_small = train_feat.loc[sample_idx, TARGET_COL].values

t0 = time.time()
lgb_models_pl, lgb_oof_pl = train_lgb(X_small, y_small, X_small, cat_features)
brier_pl = brier_score_loss(y_small, lgb_oof_pl)
print(f"[LightGBM+상성] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+상성] OOF Brier (20만행): {brier_pl:.5f}")
print(f"참고 - 상성 피처 없는 기존 LightGBM 20만행 Brier: 0.245666")
print(f"개선폭(참고용): {(0.245666 - brier_pl) / 0.245666 * 100:.4f}%")

  [LGB fold 0] brier=0.24555
  [LGB fold 1] brier=0.24573
  [LGB fold 2] brier=0.24504
  [LGB fold 3] brier=0.24572
  [LGB fold 4] brier=0.24568
[LightGBM+상성] 소요시간: 20.4초
[LightGBM+상성] OOF Brier (20만행): 0.24554
참고 - 상성 피처 없는 기존 LightGBM 20만행 Brier: 0.245666
개선폭(참고용): 0.0504%


## 3. 전체 데이터로 최종 확인 — 이 결과로 판단

In [4]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_pl_full, lgb_oof_pl_full = train_lgb(X_full, y_full, X_full, cat_features)
brier_pl_full = brier_score_loss(y_full, lgb_oof_pl_full)
print(f"[LightGBM+상성] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+상성] OOF Brier (전체): {brier_pl_full:.5f}")
print(f"참고 - 상성 피처 없는 기존 LightGBM 전체 Brier: 0.244139")
print(f"개선폭: {(0.244139 - brier_pl_full) / 0.244139 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24406
  [LGB fold 1] brier=0.24410
  [LGB fold 2] brier=0.24402
  [LGB fold 3] brier=0.24415
  [LGB fold 4] brier=0.24394
[LightGBM+상성] 소요시간: 245.6초
[LightGBM+상성] OOF Brier (전체): 0.24405
참고 - 상성 피처 없는 기존 LightGBM 전체 Brier: 0.244139
개선폭: 0.0352% (양수면 개선)


## 4. Feature Importance로 실제 활용도 확인

In [5]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_pl_full)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
print("좌우 상성 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(platoon_cols)][["rank", "feature", "mean_gain", "share_pct"]])

print(f"\n참고 - pitcher_hand/batter_hand 개별 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(["pitcher_hand", "batter_hand"])][["rank", "feature", "mean_gain", "share_pct"]])

좌우 상성 피처 순위:


,rank,feature,mean_gain,share_pct
30,31,hand_matchup,11991.650222,1.018280
37,38,is_same_hand,7989.007593,0.678393



참고 - pitcher_hand/batter_hand 개별 피처 순위:


,rank,feature,mean_gain,share_pct
44,45,batter_hand,2867.908885,0.243531
47,48,pitcher_hand,1390.337488,0.118062
